
# Customer Intelligence & Predictive Retention Platform
## Notebook 02 — Model Evaluation

### Objective
Evaluate the trained Logistic Regression model for customer churn prediction using a held-out test dataset.

### Dataset
- **Source:** IBM Telco Customer Churn dataset
- **Target variable:** `Churn`
- **Target encoding:** No = 0, Yes = 1
- **Test size:** 20%
- **Random state:** 42
- **Split strategy:** Stratified train-test split

### Evaluation Techniques
1. Accuracy
2. Precision
3. Recall
4. F1-score
5. ROC-AUC
6. Confusion matrix
7. ROC curve

### Generated Reports
- `reports/confusion_matrix.png`
- `reports/roc_curve.png`
- `reports/notebook_model_evaluation.csv`

### Interpretation Note
Accuracy alone may not be sufficient for churn prediction because the classes are imbalanced. Precision, recall, F1-score, ROC-AUC, and the confusion matrix should be reviewed together.

These metrics describe performance on the selected test split; they do not guarantee the same performance in production.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

print("Model evaluation libraries imported successfully!")


In [ ]:

from pathlib import Path
import pandas as pd

# Locate the existing model comparison report
REPORTS_DIR = Path("../reports")
MODEL_RESULTS_PATH = REPORTS_DIR / "model_comparison.csv"

if not MODEL_RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Model comparison file not found: {MODEL_RESULTS_PATH.resolve()}\n"
        "Please check the reports folder."
    )

model_results = pd.read_csv(MODEL_RESULTS_PATH)

print("MODEL COMPARISON RESULTS")
print("=" * 60)
print("Dataset shape:", model_results.shape)
print("\nAvailable columns:")
print(model_results.columns.tolist())

display(model_results)


In [ ]:

# Step 18: Compare model performance

print("Available columns:", model_results.columns.tolist())

# Automatically identify the model-name column
possible_name_cols = ["Model", "model", "Model_Name", "model_name", "Algorithm"]
model_col = next(
    (col for col in possible_name_cols if col in model_results.columns),
    None
)

if model_col is None:
    raise ValueError(
        "Model-name column not identified. Check the column names printed above."
    )

# Identify available performance metrics
metrics = [
    "Accuracy", "Precision", "Recall", "F1_Score", "F1-Score", "ROC_AUC",
    "accuracy", "precision", "recall", "f1", "f1_score", "roc_auc"
]
metric_cols = [col for col in metrics if col in model_results.columns]

if not metric_cols:
    raise ValueError(
        "No recognised metric columns found. Check the CSV column names."
    )

comparison = model_results[[model_col] + metric_cols].copy()
comparison = comparison.set_index(model_col)

# Convert metric values to numeric where possible
comparison = comparison.apply(pd.to_numeric, errors="coerce")

display(comparison)

ax = comparison.plot(kind="bar", figsize=(12, 6))
ax.set_title("Machine Learning Model Performance Comparison")
ax.set_xlabel("Model")
ax.set_ylabel("Metric Score")
ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha="right")
plt.legend(title="Metrics", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:

# Step 19: Identify the best model by F1-score

# Find the F1-score column available in the results
f1_candidates = ["F1_Score", "F1-Score", "f1_score", "f1"]
f1_col = next(
    (col for col in f1_candidates if col in model_results.columns),
    None
)

if f1_col is None:
    print("F1-score column not found.")
    print("Available columns:", model_results.columns.tolist())
else:
    # Use the model-name column identified in Step 18
    results = model_results.copy()
    results[f1_col] = pd.to_numeric(results[f1_col], errors="coerce")
    valid_results = results.dropna(subset=[f1_col])

    if valid_results.empty:
        print("No numeric F1-score values available.")
    else:
        best_row = valid_results.loc[valid_results[f1_col].idxmax()]

        print("BEST MODEL BASED ON F1-SCORE")
        print("=" * 45)
        print("Model:", best_row[model_col])
        print(f"F1-score: {best_row[f1_col]:.4f}")
        print("\nAll models ranked by F1-score:")

        display(
            valid_results[[model_col, f1_col]]
            .sort_values(f1_col, ascending=False)
            .reset_index(drop=True)
        )


In [ ]:

from pathlib import Path
import joblib

MODEL_PATH = Path("../models/churn_model.joblib")

print("Trained model file exists:", MODEL_PATH.exists())

if MODEL_PATH.exists():
    model = joblib.load(MODEL_PATH)
    print("Model loaded successfully!")
    print("Model type:", type(model).__name__)
else:
    print("Model file not found at:", MODEL_PATH.resolve())
    print("Check the models folder.")


In [ ]:

# Step 21: Load dataset and inspect the target column

DATA_PATH = Path("../data/raw/Telco-Customer-Churn.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}")

df_eval = pd.read_csv(DATA_PATH)

print("Dataset shape:", df_eval.shape)
print("\nTarget column distribution:")
display(df_eval["Churn"].value_counts(dropna=False).to_frame("Customer_Count"))

print("\nModel details:")
print(model)

print("\nDataset columns:")
print(df_eval.columns.tolist())


In [ ]:

# Step 23: Confusion Matrix for the trained Logistic Regression model

from pathlib import Path
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# Load the same cleaned dataset used by train.py
DATA_PATH = Path("../data/processed/telco_churn_cleaned.csv")
MODEL_PATH = Path("../models/churn_model.joblib")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Cleaned dataset not found: {DATA_PATH.resolve()}")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Trained model not found: {MODEL_PATH.resolve()}")

df_eval = pd.read_csv(DATA_PATH)

# Reproduce the exact target and test split from train.py
X = df_eval.drop(columns=["customerID", "Churn"])
y = df_eval["Churn"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Load the saved preprocessing + model pipeline
model = joblib.load(MODEL_PATH)
predictions = model.predict(X_test)

# Calculate confusion matrix
cm = confusion_matrix(y_test, predictions, labels=[0, 1])

print("CONFUSION MATRIX")
print("Rows = Actual | Columns = Predicted")
print("Class 0 = No Churn | Class 1 = Churn")
print(cm)

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_test,
        predictions,
        labels=[0, 1],
        target_names=["No Churn", "Churn"],
        zero_division=0
    )
)

# Plot the matrix
REPORT_DIR = Path("../reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"],
    ax=ax
)
ax.set_title("Confusion Matrix — Logistic Regression")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("Actual Label")
fig.tight_layout()

fig.savefig(
    REPORT_DIR / "confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

print("\nConfusion matrix saved to:", (REPORT_DIR / "confusion_matrix.png").resolve())


In [ ]:

# Step 24: ROC Curve and ROC-AUC

from sklearn.metrics import roc_curve, roc_auc_score

# Use the same test split and trained model from Step 23
probabilities = model.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, probabilities)
roc_auc = roc_auc_score(y_test, probabilities)

print(f"ROC-AUC Score: {roc_auc:.4f}")

REPORT_DIR = Path("../reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(
    fpr,
    tpr,
    label=f"Logistic Regression (AUC = {roc_auc:.3f})"
)
ax.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
ax.set_title("ROC Curve — Customer Churn Prediction")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
fig.tight_layout()

fig.savefig(
    REPORT_DIR / "roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

print("ROC curve saved to:", (REPORT_DIR / "roc_curve.png").resolve())


In [ ]:

# Step 25: Save model evaluation metrics to reports

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Use the test predictions and probabilities from previous steps
evaluation_metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, predictions),
    "Precision": precision_score(y_test, predictions, zero_division=0),
    "Recall": recall_score(y_test, predictions, zero_division=0),
    "F1_Score": f1_score(y_test, predictions, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, probabilities)
}

evaluation_df = pd.DataFrame([evaluation_metrics])

REPORT_DIR = Path("../reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

report_path = REPORT_DIR / "notebook_model_evaluation.csv"
evaluation_df.to_csv(report_path, index=False)

display(evaluation_df.round(4))

print("Evaluation report saved successfully!")
print("Location:", report_path.resolve())
